# StudioRoom Shadow Removal — ONNX Training
Trains `shadow_g1.onnx` (mask), `shadow_g2.onnx` (removal), `face_shadow_removal.onnx`.

**Dataset:** [ISTD](https://github.com/DeepInsight-PCALab/ST-CGAN) — download and unzip to Google Drive.

Drive folder structure expected:
```
MyDrive/ISTD_Dataset/
  train/
    shadow/        ← input shadow images
    shadow_free/   ← ground truth
    mask/          ← binary shadow mask (0=lit, 255=shadow)
  test/
    shadow/
    shadow_free/
    mask/
```

In [ ]:
# ── 1. Mount Drive & install deps ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q onnx onnxruntime

In [ ]:
# ── 2. Verify GPU ───────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ── 3. Dataset paths ────────────────────────────────────────────────────────
import os
ISTD_ROOT   = '/content/drive/MyDrive/ISTD_Dataset'
TRAIN_SHADOW      = os.path.join(ISTD_ROOT, 'train/shadow')
TRAIN_SHADOW_FREE = os.path.join(ISTD_ROOT, 'train/shadow_free')
TRAIN_MASK        = os.path.join(ISTD_ROOT, 'train/mask')
OUT_DIR     = '/content/drive/MyDrive/StudioRoom_models'
os.makedirs(OUT_DIR, exist_ok=True)
print('Train images:', len(os.listdir(TRAIN_SHADOW)))

In [ ]:
# ── 4. Model architecture (same as app) ─────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class ConvBnRelu(nn.Sequential):
    def __init__(self, cin, cout, k=3, s=1, p=1):
        super().__init__(
            nn.Conv2d(cin, cout, k, s, p, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

class ChannelAttn(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, max(ch//r, 4)), nn.ReLU(inplace=True),
            nn.Linear(max(ch//r, 4), ch), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x).view(x.shape[0], x.shape[1], 1, 1)

class EncBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Sequential(ConvBnRelu(cin, cout), ConvBnRelu(cout, cout))
        self.attn = ChannelAttn(cout)
    def forward(self, x): return self.attn(self.conv(x))

class DecBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.up   = nn.ConvTranspose2d(cin, cout, 2, 2)
        self.conv = nn.Sequential(ConvBnRelu(cout*2, cout), ConvBnRelu(cout, cout))
        self.attn = ChannelAttn(cout)
    def forward(self, x, skip):
        x = self.up(x)
        return self.attn(self.conv(torch.cat([x, skip], 1)))

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, base=32):
        super().__init__()
        b = base
        self.e1 = EncBlock(in_ch, b);   self.e2 = EncBlock(b, b*2)
        self.e3 = EncBlock(b*2, b*4);   self.pool = nn.MaxPool2d(2)
        self.bot = nn.Sequential(ConvBnRelu(b*4, b*8), ConvBnRelu(b*8, b*8))
        self.d3 = DecBlock(b*8, b*4);   self.d2 = DecBlock(b*4, b*2)
        self.d1 = DecBlock(b*2, b);     self.head = nn.Conv2d(b, out_ch, 1)
    def forward(self, x):
        s1=self.e1(x); s2=self.e2(self.pool(s1)); s3=self.e3(self.pool(s2))
        x=self.bot(self.pool(s3))
        x=self.d3(x,s3); x=self.d2(x,s2); x=self.d1(x,s1)
        return torch.tanh(self.head(x))

print('Architecture OK')

In [ ]:
# ── 5. Dataset loaders ──────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

SIZE = 256
to_tensor = transforms.Compose([transforms.Resize((SIZE,SIZE)), transforms.ToTensor()])
normalize = transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])  # [0,1]→[-1,1]

class ISTDDataset(Dataset):
    def __init__(self, shadow_dir, shadow_free_dir, mask_dir):
        self.names = sorted(os.listdir(shadow_dir))
        self.s  = shadow_dir
        self.sf = shadow_free_dir
        self.m  = mask_dir
    def __len__(self): return len(self.names)
    def __getitem__(self, idx):
        n = self.names[idx]
        shadow      = normalize(to_tensor(Image.open(os.path.join(self.s,  n)).convert('RGB')))
        shadow_free = normalize(to_tensor(Image.open(os.path.join(self.sf, n)).convert('RGB')))
        mask        = to_tensor(Image.open(os.path.join(self.m,  n)).convert('L'))  # 1ch [0,1]
        # ISTD mask: 255=shadow → invert to 1=shadow
        mask = (mask > 0.5).float()
        return shadow, shadow_free, mask

ds    = ISTDDataset(TRAIN_SHADOW, TRAIN_SHADOW_FREE, TRAIN_MASK)
loader = DataLoader(ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset: {len(ds)} pairs, {len(loader)} batches/epoch')

In [ ]:
# ── 6. Train G1 (shadow mask) ───────────────────────────────────────────────
G1_EPOCHS = 60

g1 = UNet(in_ch=3, out_ch=1, base=32).to(DEVICE)
opt_g1 = torch.optim.Adam(g1.parameters(), lr=2e-4)
sch_g1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g1, G1_EPOCHS)
bce = nn.BCEWithLogitsLoss()

for ep in range(1, G1_EPOCHS+1):
    g1.train(); total = 0
    for shadow, _, mask in loader:
        shadow, mask = shadow.to(DEVICE), mask.to(DEVICE)
        pred = g1(shadow)          # tanh output → use raw logit before tanh
        # Use L1 on tanh output vs mask (mask is 0/1, tanh maps to -1/+1)
        mask_t = mask * 2 - 1      # [0,1] → [-1,+1]
        loss = F.l1_loss(pred, mask_t)
        opt_g1.zero_grad(); loss.backward(); opt_g1.step()
        total += loss.item()
    sch_g1.step()
    if ep % 10 == 0:
        print(f'G1 ep {ep}/{G1_EPOCHS}  loss={total/len(loader):.4f}')

torch.save(g1.state_dict(), '/content/g1.pth')
print('G1 training done')

In [ ]:
# ── 7. Train G2 (shadow removal) ────────────────────────────────────────────
G2_EPOCHS = 120

g1.eval()
g2 = UNet(in_ch=4, out_ch=3, base=32).to(DEVICE)
opt_g2 = torch.optim.Adam(g2.parameters(), lr=2e-4)
sch_g2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g2, G2_EPOCHS)

for ep in range(1, G2_EPOCHS+1):
    g2.train(); total = 0
    for shadow, shadow_free, _ in loader:
        shadow, shadow_free = shadow.to(DEVICE), shadow_free.to(DEVICE)
        with torch.no_grad():
            mask_pred = g1(shadow)          # [-1,+1] tanh mask
        x4 = torch.cat([shadow, mask_pred], dim=1)   # 4-channel input
        out = g2(x4)
        loss = F.l1_loss(out, shadow_free)
        opt_g2.zero_grad(); loss.backward(); opt_g2.step()
        total += loss.item()
    sch_g2.step()
    if ep % 20 == 0:
        print(f'G2 ep {ep}/{G2_EPOCHS}  loss={total/len(loader):.4f}')

torch.save(g2.state_dict(), '/content/g2.pth')
print('G2 training done')

In [ ]:
# ── 8. Train face_shadow_removal (single-stage, same ISTD data) ─────────────
# For portrait-specific results, replace ISTD with a face shadow dataset.
# Here we train a single-stage model as a simpler alternative.
FACE_EPOCHS = 100

face = UNet(in_ch=3, out_ch=3, base=32).to(DEVICE)
opt_face = torch.optim.Adam(face.parameters(), lr=2e-4)
sch_face = torch.optim.lr_scheduler.CosineAnnealingLR(opt_face, FACE_EPOCHS)

for ep in range(1, FACE_EPOCHS+1):
    face.train(); total = 0
    for shadow, shadow_free, _ in loader:
        shadow, shadow_free = shadow.to(DEVICE), shadow_free.to(DEVICE)
        out  = face(shadow)
        loss = F.l1_loss(out, shadow_free)
        opt_face.zero_grad(); loss.backward(); opt_face.step()
        total += loss.item()
    sch_face.step()
    if ep % 20 == 0:
        print(f'Face ep {ep}/{FACE_EPOCHS}  loss={total/len(loader):.4f}')

torch.save(face.state_dict(), '/content/face.pth')
print('Face shadow training done')

In [ ]:
# ── 9. Export all 3 to ONNX (inline, no sidecar) ───────────────────────────
import io, onnx

def export_inline(model, dummy, path, in_names, out_names):
    model.eval().cpu()
    buf = io.BytesIO()
    torch.onnx.export(model, dummy.cpu(), buf,
                      input_names=in_names, output_names=out_names,
                      opset_version=17, dynamo=False)
    buf.seek(0)
    proto = onnx.load_from_string(buf.read())
    onnx.save(proto, path, save_as_external_data=False)
    print(f'  {path}  ({os.path.getsize(path)//1024} KB)')

dummy3 = torch.zeros(1,3,256,256)
dummy4 = torch.zeros(1,4,256,256)

g1.load_state_dict(torch.load('/content/g1.pth', map_location='cpu'))
g2.load_state_dict(torch.load('/content/g2.pth', map_location='cpu'))
face.load_state_dict(torch.load('/content/face.pth', map_location='cpu'))

export_inline(g1,   dummy3, os.path.join(OUT_DIR,'shadow_g1.onnx'),         ['rgb_input'],      ['shadow_mask'])
export_inline(g2,   dummy4, os.path.join(OUT_DIR,'shadow_g2.onnx'),         ['rgb_mask_input'], ['rgb_out'])
export_inline(face, dummy3, os.path.join(OUT_DIR,'face_shadow_removal.onnx'),['rgb_input'],      ['rgb_out'])

print('\nAll models saved to:', OUT_DIR)
print('Download and drop into: feature/photo-editor/src/main/assets/models/')

In [ ]:
# ── 10. Quick visual sanity check ───────────────────────────────────────────
import matplotlib.pyplot as plt

g1.eval(); g2.eval()
shadow, shadow_free, mask = ds[0]
with torch.no_grad():
    inp = shadow.unsqueeze(0)
    m   = g1(inp)
    out = g2(torch.cat([inp, m], 1)).squeeze(0)

def show(t, title):
    img = ((t.permute(1,2,0).numpy() * 0.5 + 0.5) * 255).clip(0,255).astype('uint8')
    plt.imshow(img); plt.title(title); plt.axis('off')

plt.figure(figsize=(15,4))
plt.subplot(1,4,1); show(shadow, 'Input (shadow)')
plt.subplot(1,4,2); plt.imshow(m.squeeze().numpy()*0.5+0.5, cmap='gray'); plt.title('G1 mask'); plt.axis('off')
plt.subplot(1,4,3); show(out, 'G2 output')
plt.subplot(1,4,4); show(shadow_free, 'Ground truth')
plt.tight_layout(); plt.show()